In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/non_exploratory/check'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/adaptive_bp_repair_paths.json')


In [ ]:
from decoders.cpp_backend import AdaptiveBP
from codes.ca_code import CA_9_2_5
from noises_ca import CircuitBuilder, ErrorRate
import numpy as np
import networkx as nx
from utils.misc import *
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import visualize_det_on_tanner_graph


In [ ]:
code = CA_9_2_5()
p = 0.0002848035868435


In [ ]:
error_rate = ErrorRate(p=p)
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()


## Weight-2 Error

### Case 1

error = [1, 9], [2, 7]

In [ ]:
adaptive_bp = AdaptiveBP(
    code=code,
    error_rate=p,
    max_iter=60,
    bp_method='minimum_sum',
    adaptive_k=8,
)


In [ ]:
syndrome = np.asarray(str2bits('11011000000110'))
compiled_b = adaptive_bp.compile_decoder_for_dem(dem=dem)
corr = compiled_b.decoder.decode_to_correction(syndrome)
corr


In [ ]:
bits2id(corr)


In [ ]:
(compiled_b.decoder.bp.log_prob_ratios, compiled_b.decoder.bp.auxiliary_log_prob_ratios)


In [ ]:
(compiled_b.decoder.converge(), compiled_b.decoder.auxiliary_converge())


### Case 2

error = [1, 7], [2, 9]

In [ ]:
adaptive_bp = AdaptiveBP(
    code=code,
    error_rate=p,
    max_iter=60,
    bp_method='minimum_sum',
    adaptive_k=6,
)


In [ ]:
syndrome = np.asarray(str2bits('11010000000010'))
compiled_b = adaptive_bp.compile_decoder_for_dem(dem=dem)
corr = compiled_b.decoder.decode_to_correction(syndrome)
corr


In [ ]:
bits2id(corr)


In [ ]:
(compiled_b.decoder.bp.log_prob_ratios, compiled_b.decoder.bp.auxiliary_log_prob_ratios)


In [ ]:
(compiled_b.decoder.converge(), compiled_b.decoder.auxiliary_converge())


## Comparison: Adaptive BP vs. BP+OSD

In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

DATA_DIR = _data_paths.run()
DATA_DIR.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD, CythonBPOSD, AdaptiveBP
from src.decoders import Tesseract
from beliefmatching import detector_error_model_to_check_matrices
from utils.graph_vis import visualize_det_on_tanner_graph
from utils.graph_struct import build_tanner_graph_from_dem


In [ ]:
ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
WEIGHT_DIR = _data_paths.input('error_weight')
os.makedirs(WEIGHT_DIR, exist_ok=True)
num_errs = 33
num_dets = 14
num_obs = 2
max_iter = 60
code = ca_codes['CA_9_2_5']


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2_data = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=types)


In [ ]:
weight2_data[(weight2_data['pred_b'] != weight2_data['true']) & (weight2_data['pred_t'] == weight2_data['true'])]


In [ ]:
fails = 0
fail_cases = []
total = weight2_data.shape[0]

for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    e = str2bits(row['err'])
    syndrome = str2bits(row['det'])
    true_observable = str2bits(row['true'])
    decoder = AdaptiveBP(
        code=code,
        error_rate=float(row['p']),
        max_iter=60,
        bp_method='minimum_sum',
        adaptive_k=8,
    )
    compiled_b = decoder.compile_decoder_for_dem(dem=dem)
    pred = compiled_b.decoder.decode(syndrome)

    if not np.array_equal(pred, true_observable):
        fails += 1
        fail_cases.append(row)

print(f'Failed: {fails}/{total}')


In [ ]:
pd.DataFrame(fail_cases)
